# AI Model Validation: From Mathematics to Trust
## Episode 08 — Thresholds, ROC, Precision–Recall, and the Mathematics of Decision Trade-Offs

**Author: Vishwanath Akuthota**  
**Founder of DrPinnacle, OpenVals and The Foundry's**

Episode 07 showed that accuracy alone can conceal catastrophic classification behaviour.

Episode 08 asks a deeper question:

> **Once a model produces a score, where should we place the decision threshold?**

There is no universally best threshold. The answer depends on prevalence, error costs, operating constraints, probability calibration, and the decision we are trying to make.

This episode develops threshold mathematics, ROC geometry, AUC, precision–recall curves, likelihood ratios, Bayes decision rules, expected cost, Youden's J, constrained threshold selection, calibration, and operating-point validation.


## 1. A Score Is Not Yet a Decision

Many classifiers produce a continuous score rather than a final class.

### s(x) ∈ ℝ

or, for a probability-producing system:

### p̂(x) ∈ [0, 1]

A threshold τ transforms the score into a decision:

### Ŷτ = 1[s(x) ≥ τ]

The model score and the deployment decision are therefore different objects.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import platform, sys

print("Python :",sys.version.split()[0])
print("System :",platform.system(),platform.release())
print("Machine:",platform.machine())
print("NumPy  :",np.__version__)


## 2. Thresholds Create Confusion Matrices

For a fixed dataset and score-producing model:

### CM = CM(τ)

Therefore:

### TP = TP(τ)
### FN = FN(τ)
### FP = FP(τ)
### TN = TN(τ)

Every threshold-dependent classification metric inherits this dependence.


## 3. Build a Controlled Classification Experiment

We simulate positive and negative score distributions. The experiment is deliberately transparent so that the effect of threshold movement can be observed directly.


In [ ]:
rng=np.random.default_rng(808)

n_pos=500
n_neg=2500

positive_scores=np.clip(rng.normal(.72,.17,n_pos),0,1)
negative_scores=np.clip(rng.normal(.28,.18,n_neg),0,1)

scores=np.concatenate([positive_scores,negative_scores])
y=np.concatenate([np.ones(n_pos,dtype=int),np.zeros(n_neg,dtype=int)])

print("N:",len(y))
print("Positive prevalence:",y.mean())

plt.figure(figsize=(8,5))
plt.hist(negative_scores,bins=35,alpha=.55,label="Actual negative")
plt.hist(positive_scores,bins=35,alpha=.55,label="Actual positive")
plt.xlabel("Model score")
plt.ylabel("Frequency")
plt.title("Overlapping Score Distributions")
plt.legend()
plt.grid(alpha=.2)
plt.show()


The overlap is where classification errors become unavoidable. Moving the threshold changes which type of error we accept.


## 4. Confusion Matrix as a Function of τ


In [ ]:
def confusion(y,s,t):
    pred=(s>=t).astype(int)
    tp=np.sum((y==1)&(pred==1))
    fn=np.sum((y==1)&(pred==0))
    fp=np.sum((y==0)&(pred==1))
    tn=np.sum((y==0)&(pred==0))
    return tp,fn,fp,tn

def metric_set(tp,fn,fp,tn):
    precision=tp/(tp+fp) if tp+fp else np.nan
    recall=tp/(tp+fn) if tp+fn else np.nan
    specificity=tn/(tn+fp) if tn+fp else np.nan
    fpr=fp/(fp+tn) if fp+tn else np.nan
    accuracy=(tp+tn)/(tp+fn+fp+tn)
    f1=2*precision*recall/(precision+recall) if np.isfinite(precision) and precision+recall else np.nan
    return precision,recall,specificity,fpr,accuracy,f1

for t in [.20,.35,.50,.65,.80]:
    cm=confusion(y,scores,t)
    vals=metric_set(*cm)
    print(f"τ={t:.2f}  TP={cm[0]:3d} FN={cm[1]:3d} FP={cm[2]:4d} TN={cm[3]:4d} "
          f"P={vals[0]:.3f} R={vals[1]:.3f}")


## 5. Lower Thresholds and Higher Thresholds

A lower threshold generally predicts more positives:

### τ ↓ ⇒ TP ↑ and FP ↑

A higher threshold generally predicts fewer positives:

### τ ↑ ⇒ FP ↓ and TP ↓

Equivalently, false negatives usually rise as the threshold rises.

This is not a defect of the metric. It is the geometry of overlapping score distributions.


## 6. Sensitivity and Specificity Across Thresholds

### Sensitivity(τ) = TP(τ) / [TP(τ) + FN(τ)]

### Specificity(τ) = TN(τ) / [TN(τ) + FP(τ)]


In [ ]:
thresholds=np.linspace(0,1,301)
sensitivity=[]
specificity=[]
precision=[]
fpr=[]
f1=[]

for t in thresholds:
    tp,fn,fp,tn=confusion(y,scores,t)
    p,r,sp,fr,a,f=metric_set(tp,fn,fp,tn)
    precision.append(p); sensitivity.append(r); specificity.append(sp)
    fpr.append(fr); f1.append(f)

plt.figure(figsize=(8,5))
plt.plot(thresholds,sensitivity,label="Sensitivity / Recall")
plt.plot(thresholds,specificity,label="Specificity")
plt.xlabel("Threshold τ")
plt.ylabel("Metric")
plt.title("Sensitivity–Specificity Trade-Off")
plt.legend()
plt.grid(alpha=.25)
plt.show()


A threshold cannot be evaluated independently of the trade-off it creates.


## 7. ROC Geometry

ROC space uses:

### x-axis = FPR = FP / (FP + TN)

### y-axis = TPR = TP / (TP + FN)

Each threshold corresponds to one point:

### ROC(τ) = [FPR(τ), TPR(τ)]

Sweeping τ traces the ROC curve.


In [ ]:
order=np.argsort(fpr)
roc_fpr=np.array(fpr)[order]
roc_tpr=np.array(sensitivity)[order]

plt.figure(figsize=(7,6))
plt.plot(roc_fpr,roc_tpr,label="ROC")
plt.plot([0,1],[0,1],linestyle="--",label="Random-ranking reference")
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC Geometry")
plt.legend()
plt.grid(alpha=.25)
plt.show()


## 8. Meaning of the Diagonal

A ranking system with no discrimination has, in expectation:

### TPR ≈ FPR

This produces the diagonal reference line.

A useful ranking system tends to push the ROC curve toward the upper-left region:

### FPR → 0
### TPR → 1


## 9. ROC AUC

Area Under the ROC Curve is:

### AUC = ∫₀¹ TPR(FPR) d(FPR)

A powerful probabilistic interpretation is:

### AUC = P(S⁺ > S⁻) + ½P(S⁺ = S⁻)

where S⁺ is a randomly selected positive score and S⁻ a randomly selected negative score.


In [ ]:
pair_greater=(positive_scores[:,None]>negative_scores[None,:]).mean()
pair_equal=(positive_scores[:,None]==negative_scores[None,:]).mean()
auc_pair=pair_greater+.5*pair_equal

auc_trap=np.trapz(roc_tpr,roc_fpr)

print("Pairwise AUC   :",auc_pair)
print("Integrated AUC :",auc_trap)


AUC measures ranking discrimination across thresholds.

It does **not** tell us:

- which threshold to deploy
- whether predicted probabilities are calibrated
- what false positives cost
- what false negatives cost
- whether performance is acceptable for a specific application


## 10. AUC Is Threshold-Free, Decisions Are Not

### Ranking Quality ≠ Decision Quality

Two systems can have similar AUC yet produce materially different outcomes at the operating region that matters.

Validation must therefore inspect the intended operating point, not only the global area.


## 11. Youden's J Statistic

One classical threshold criterion is:

### J(τ) = Sensitivity(τ) + Specificity(τ) − 1

Because:

### Specificity = 1 − FPR

we can also write:

### J(τ) = TPR(τ) − FPR(τ)

The threshold maximizing J maximizes vertical separation from the ROC diagonal.


In [ ]:
J=np.array(sensitivity)+np.array(specificity)-1
j_idx=np.nanargmax(J)
j_threshold=thresholds[j_idx]

print("Threshold maximizing Youden J:",j_threshold)
print("J:",J[j_idx])
print("Sensitivity:",sensitivity[j_idx])
print("Specificity:",specificity[j_idx])


Youden's J treats sensitivity and specificity symmetrically.

That may be useful in some settings, but it does not encode unequal error costs or prevalence-dependent consequences.

### Maximum J ≠ Universal Best Threshold


## 12. Precision–Recall Geometry

Precision:

### Precision = TP / (TP + FP)

Recall:

### Recall = TP / (TP + FN)

A precision–recall curve plots precision against recall as τ varies.


In [ ]:
P=np.array(precision)
R=np.array(sensitivity)
valid=np.isfinite(P)

plt.figure(figsize=(7,6))
plt.plot(R[valid],P[valid],label="PR curve")
plt.axhline(y.mean(),linestyle="--",label="Prevalence reference")
plt.xlabel("Recall")
plt.ylabel("Precision")
plt.title("Precision–Recall Geometry")
plt.legend()
plt.grid(alpha=.25)
plt.show()


## 13. Why Prevalence Appears in Precision

Let:

### π = P(Y = 1)

Then, by Bayes' rule:

### Precision =
### [TPR × π] / [TPR × π + FPR × (1 − π)]

Precision therefore changes when prevalence changes even if TPR and FPR remain fixed.


## 14. Experiment — Same Classifier, Different Populations


In [ ]:
tpr_fixed=.90
fpr_fixed=.05

for prev in [.50,.20,.05,.01,.001]:
    ppv=(tpr_fixed*prev)/(tpr_fixed*prev+fpr_fixed*(1-prev))
    print(f"Prevalence={prev:7.3%} -> Precision={ppv:7.3%}")


This is why a model validated in one population may produce very different positive-prediction reliability in another population.

### Same TPR + Same FPR ≠ Same Precision when prevalence changes


## 15. ROC vs Precision–Recall

ROC asks how well positives are ranked relative to negatives across thresholds.

Precision–recall emphasizes the quality and coverage of positive predictions.

When positives are rare, a small FPR can still create many false positives because the negative population is large.

Neither curve is universally superior. The validation question determines which view is more informative.


## 16. F1 as a Threshold Objective

### F1(τ) = 2 × Precision(τ) × Recall(τ) /
### [Precision(τ) + Recall(τ)]

A threshold can be selected to maximize observed F1.


In [ ]:
f1_arr=np.array(f1)
f1_idx=np.nanargmax(f1_arr)
print("Observed F1-max threshold:",thresholds[f1_idx])
print("Observed max F1:",f1_arr[f1_idx])


But maximizing F1 implicitly treats precision and recall through the harmonic mean and ignores TN.

### Maximum F1 ≠ Universal Best Threshold


## 17. Fβ Score

If recall should receive more or less weight, the Fβ family is:

### Fβ = (1 + β²)PR / (β²P + R)

For:

### β > 1

recall receives greater weight.

For:

### β < 1

precision receives greater weight.


In [ ]:
def fbeta(p,r,beta):
    if not np.isfinite(p) or beta*beta*p+r==0: return np.nan
    return (1+beta**2)*p*r/(beta**2*p+r)

for beta in [.5,1,2]:
    vals=np.array([fbeta(p,r,beta) for p,r in zip(P,R)])
    idx=np.nanargmax(vals)
    print(f"beta={beta}: best threshold={thresholds[idx]:.3f}, Fbeta={vals[idx]:.3f}")


Metric weighting can express priorities, but the choice of β itself requires domain justification.


## 18. Expected Cost

Suppose:

- false positive cost = CFP
- false negative cost = CFN

For an evaluation sample:

### Total Cost(τ) = CFP × FP(τ) + CFN × FN(τ)

A normalized expected cost can be expressed as:

### Risk(τ) = CFP × FPR(τ) × P(Y=0)
### + CFN × FNR(τ) × P(Y=1)


In [ ]:
cost_cases=[
    ("Equal",1,1),
    ("FN severe",1,20),
    ("FP severe",20,1)
]

for label,c_fp,c_fn in cost_cases:
    costs=[]
    for t in thresholds:
        tp,fn,fp,tn=confusion(y,scores,t)
        costs.append(c_fp*fp+c_fn*fn)
    idx=np.argmin(costs)
    print(f"{label:<10}: threshold={thresholds[idx]:.3f}, cost={costs[idx]}")


The preferred operating point changes because the decision objective changed—not because the classifier changed.


## 19. Bayes Decision Rule for Calibrated Probabilities

Suppose p(x) is the calibrated probability that Y=1.

Assume zero cost for correct decisions, false-positive cost CFP and false-negative cost CFN.

Predict positive when expected cost of predicting positive is lower:

### CFP × [1 − p(x)] < CFN × p(x)

Solving:

### p(x) > CFP / (CFP + CFN)

Therefore the Bayes cost-optimal probability threshold is:

### τ* = CFP / (CFP + CFN)

under these assumptions.


In [ ]:
for c_fp,c_fn in [(1,1),(1,9),(9,1),(1,99)]:
    threshold=c_fp/(c_fp+c_fn)
    print(f"CFP={c_fp:>2}, CFN={c_fn:>2} -> theoretical calibrated-probability threshold={threshold:.3f}")


This derivation is powerful, but only when the score is a meaningful calibrated probability and the assumed costs correctly represent the decision problem.


## 20. Why 0.5 Is Not Sacred

For equal FP and FN costs with calibrated probabilities:

### τ* = 0.5

But if false negatives cost nine times as much as false positives:

### τ* = 1 / (1 + 9) = 0.1

Therefore:

### Default Threshold 0.5 ≠ Scientifically Optimal Threshold


## 21. Likelihood Ratios

Sensitivity and specificity can be combined into likelihood ratios.

Positive likelihood ratio:

### LR+ = Sensitivity / (1 − Specificity)

### LR+ = TPR / FPR

Negative likelihood ratio:

### LR− = (1 − Sensitivity) / Specificity

### LR− = FNR / TNR

Likelihood ratios describe how an observed classification result changes evidence about class membership.


## 22. From Prior Odds to Posterior Odds

Bayesian updating can be expressed as:

### Posterior Odds = Prior Odds × Likelihood Ratio

where:

### Prior Odds = P(Y=1) / P(Y=0)

For a positive result:

### Posterior Odds+ = Prior Odds × LR+

This mathematically connects base rates with diagnostic evidence.


In [ ]:
sensitivity_demo=.90
specificity_demo=.95
lr_plus=sensitivity_demo/(1-specificity_demo)

for prevalence in [.50,.10,.01,.001]:
    prior_odds=prevalence/(1-prevalence)
    posterior_odds=prior_odds*lr_plus
    posterior_prob=posterior_odds/(1+posterior_odds)
    print(f"Prevalence={prevalence:.3%} -> P(Y=1 | positive)={posterior_prob:.3%}")


Even strong likelihood evidence can produce modest posterior probability when the event is extremely rare.


## 23. Threshold Constraints

Sometimes the objective is not to maximize a metric but satisfy a requirement.

Examples:

### Recall ≥ 0.95

or:

### FPR ≤ 0.01

or:

### Precision ≥ 0.90

Threshold selection then becomes a constrained decision problem.


In [ ]:
# Find highest threshold retaining at least 95% recall
R_arr=np.array(R)
eligible=np.where(R_arr>=.95)[0]
if len(eligible):
    idx=eligible[-1]
    print("Threshold with recall >= .95:",thresholds[idx])
    print("Recall:",R_arr[idx])
    print("Precision:",P[idx])
    print("FPR:",fpr[idx])

# Find lowest threshold achieving FPR <= .01
FPR_arr=np.array(fpr)
eligible=np.where(FPR_arr<=.01)[0]
if len(eligible):
    idx=eligible[0]
    print("\nThreshold with FPR <= .01:",thresholds[idx])
    print("FPR:",FPR_arr[idx])
    print("Recall:",R_arr[idx])
    print("Precision:",P[idx])


A constraint can be more interpretable than an abstract global optimum because it directly expresses an operational requirement.


## 24. Calibration

Discrimination and calibration are different.

A model can rank positives above negatives well while producing probability values that do not correspond to observed frequencies.

Calibration asks:

> Among predictions near probability q, does the event occur approximately q of the time?

Ideally:

### P(Y=1 | p̂ ≈ q) ≈ q


## 25. Experiment — Reliability Diagram

We create an intentionally distorted probability scale while preserving ranking.


In [ ]:
# Convert simulated score to a deliberately overconfident probability-like score.
p_uncal=np.clip(scores**0.55,0,1)

bins=np.linspace(0,1,11)
centers=[]
observed=[]
predicted=[]

for lo,hi in zip(bins[:-1],bins[1:]):
    mask=(p_uncal>=lo)&(p_uncal<hi if hi<1 else p_uncal<=hi)
    if mask.sum()>0:
        centers.append((lo+hi)/2)
        predicted.append(p_uncal[mask].mean())
        observed.append(y[mask].mean())

plt.figure(figsize=(7,6))
plt.plot([0,1],[0,1],linestyle="--",label="Perfect calibration")
plt.plot(predicted,observed,marker="o",label="Observed reliability")
plt.xlabel("Mean predicted probability")
plt.ylabel("Observed positive frequency")
plt.title("Reliability Diagram")
plt.legend()
plt.grid(alpha=.25)
plt.show()


## 26. Brier Score

For probability predictions p̂ᵢ and binary outcomes yᵢ:

### Brier = (1/n) Σᵢ₌₁ⁿ (p̂ᵢ − yᵢ)²

Lower is better.

The Brier score evaluates probabilistic prediction quality, but its interpretation still depends on prevalence and the forecasting context.


In [ ]:
brier=np.mean((p_uncal-y)**2)
print("Brier score:",brier)


## 27. Calibration Matters for Cost-Based Thresholds

The threshold:

### τ* = CFP / (CFP + CFN)

assumes calibrated probabilities.

If scores are merely ranking scores, applying a probability-derived threshold can be invalid.

### Good Ranking ≠ Good Calibration


## 28. Threshold Selection Can Overfit

Suppose hundreds of thresholds are tested on the same validation data and the best observed threshold is selected.

That threshold is itself a fitted choice.

Therefore:

### Threshold Selection = Model Selection

The selected operating point should be evaluated on data not used to optimize it.


## 29. Train, Tune, Test

A defensible workflow is:

### Train Model → Tune Threshold → Final Test

The final test set should evaluate the model **and its already-fixed threshold**.

Otherwise the reported performance can be optimistic.


## 30. Threshold Uncertainty

A threshold chosen from finite data can change when the validation sample changes.

Therefore:

### τ̂* ≠ Fixed Population Truth

Resampling can help study the stability of the selected threshold.


In [ ]:
rng2=np.random.default_rng(81)
selected=[]

for _ in range(300):
    idx=rng2.integers(0,len(y),len(y))
    yb=y[idx]; sb=scores[idx]
    values=[]
    for t in thresholds:
        tp,fn,fp,tn=confusion(yb,sb,t)
        p,r,sp,fr,a,f=metric_set(tp,fn,fp,tn)
        values.append(f)
    selected.append(thresholds[np.nanargmax(values)])

selected=np.array(selected)
print("Median selected F1 threshold:",np.median(selected))
print("10th–90th percentile:",np.quantile(selected,[.10,.90]))

plt.figure(figsize=(8,5))
plt.hist(selected,bins=30)
plt.xlabel("Bootstrap-selected threshold")
plt.ylabel("Frequency")
plt.title("Threshold Selection Has Sampling Variability")
plt.grid(alpha=.2)
plt.show()


The threshold itself has uncertainty. Reporting a threshold without evidence of its stability can create false precision.


## 31. Distribution Shift Can Move the Best Operating Point

If prevalence, score distributions, costs, or consequences change after deployment:

### τ*deployment may differ from τ*validation

Thresholds may therefore require monitoring and controlled recalibration or re-selection.


## 32. Thresholds and High-Stakes Systems

For high-stakes decisions, one global threshold may be insufficient.

A system may instead use:

- an automatic-negative region
- an automatic-positive region
- an uncertainty or human-review region

Conceptually:

### s < τlow → Negative
### τlow ≤ s < τhigh → Review
### s ≥ τhigh → Positive

This turns binary automation into selective decision-making.


## 33. Abstention and Selective Prediction

Let the model abstain when confidence is insufficient.

Coverage:

### Coverage = Fraction of cases receiving automatic decisions

Selective risk:

### Selective Risk = Error among automatically decided cases

Reducing coverage can sometimes reduce risk by routing uncertain cases elsewhere.

The correct trade-off depends on operational capacity and consequence.


## 34. Threshold Validation Is Contextual

A defensible threshold report should specify:

- score definition
- probability calibration status
- positive class
- prevalence
- threshold
- confusion matrix at threshold
- sensitivity and specificity
- precision and recall
- FPR and FNR
- cost assumptions
- operational constraints
- uncertainty of metrics
- stability of selected threshold
- subgroup operating behaviour
- deployment distribution


## 35. OpenVals Perspective

A validation platform should distinguish three layers:

### Layer 1 — Model Score Quality
Ranking, discrimination, probability quality.

### Layer 2 — Decision Policy
Thresholds, abstention rules, constraints, cost functions.

### Layer 3 — Deployment Evidence
Observed outcomes, drift, subgroup behaviour, threshold stability and changing prevalence.

Trust cannot be assigned to the model score alone because real-world harm or value emerges from the decision policy applied to that score.


## 36. Scientific Decision Chain

### Model → Score → Calibration → Threshold → Decision → Error Trade-Off → Cost → Evidence → Trust Decision

This is the bridge from statistical prediction to operational decision-making.


## 37. Mathematical Summary

### Ŷτ = 1[s(x) ≥ τ]

### TPR = TP / (TP + FN)

### FPR = FP / (FP + TN)

### J = TPR − FPR

### AUC = P(S⁺ > S⁻) + ½P(S⁺ = S⁻)

### Precision = [TPR × π] / [TPR × π + FPR × (1 − π)]

### Fβ = (1 + β²)PR / (β²P + R)

### Risk = CFP × FPR × P(Y=0) + CFN × FNR × P(Y=1)

### τ* = CFP / (CFP + CFN)

### LR+ = TPR / FPR

### LR− = FNR / TNR

### Posterior Odds = Prior Odds × Likelihood Ratio

### Brier = (1/n) Σ(p̂ᵢ − yᵢ)²


## 38. Central Conclusion

### Best Threshold ≠ Universal Constant

> **A threshold is not where the model becomes correct. It is where we choose how the model will make mistakes.**

ROC describes discrimination.

Precision–recall exposes positive-class trade-offs.

AUC measures ranking quality.

Calibration asks whether probabilities mean what they claim.

Decision theory connects probabilities to consequences.

The scientific objective is not to maximize a convenient metric. It is to choose and validate an operating policy whose errors, uncertainty, and consequences are acceptable for the intended use.

### Next Episode

**Episode 09 — Probability Calibration: Can We Trust the Model's Confidence?**

Episode 09 can go deeper into probability semantics, calibration curves, Brier decomposition, log loss, Expected Calibration Error, overconfidence, underconfidence, temperature scaling, Platt-style calibration, isotonic calibration, and why a model can be accurate yet dangerously miscalibrated.
